<a href="https://colab.research.google.com/github/Jinan-Bark/H-Copilot/blob/master/notebooks/patient_flow_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# H-Copilot Flow Prediction — Clean Final Build
This notebook rebuilds the final 15-feature blended model (Random Forest + Neural Network) directly, skipping all the rejected experiments. Upload your `ED_triage.csv` when prompted, then Run All.

## 1. Imports and upload data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload ED_triage.csv when prompted

Saving ED_triage.csv to ED_triage.csv


## 2. Load and clean raw data

In [ ]:
triage = pd.read_csv('ED_triage.csv')

# Remove duplicate triage_code rows (re-triage records of same visit), keep first
triage = triage.drop_duplicates(subset='triage_code', keep='first')

# Build proper date column
triage['date'] = pd.to_datetime(
    triage[['admission_year','admission_month','admission_day']]
    .rename(columns={'admission_year':'year','admission_month':'month','admission_day':'day'})
)

# day_of_week: 0=Sunday ... 6=Saturday (verified against real calendar earlier in project)
triage['day_of_week'] = (triage['date'].dt.dayofweek + 1) % 7

# Build hour_slot (0-5) from admission_hour — not pre-existing in the raw CSV
triage['hour_slot'] = triage['admission_hour'] // 4

print(f'Rows after dedup: {len(triage)}')
print(triage[['date','day_of_week','hour_slot']].head())

Rows after dedup: 143346
        date  day_of_week  hour_slot
0 2017-03-21            2          0
1 2017-03-21            2          0
2 2017-03-21            2          0
3 2017-03-21            2          0
4 2017-03-21            2          0


## 3. Keep only the clean, COVID-free continuous range
Data confirmed normal through Feb 21, 2020 (Iran's first COVID cases announced Feb 19, 2020; sharp arrival drop began Feb 22, 2020).

In [ ]:
triage = triage[triage['date'] <= '2020-02-21'].copy()
print(f'Rows in clean range: {len(triage)}')
print(f'Date range: {triage["date"].min()} to {triage["date"].max()}')

Rows in clean range: 91490
Date range: 2017-03-21 00:00:00 to 2020-02-21 00:00:00


## 4. Build the slot-level patient_count table (the base for flow prediction)

In [ ]:
base = (
    triage.groupby(['date','day_of_week','hour_slot'])
    .size()
    .reset_index(name='patient_count')
)
base = base.sort_values(['date','hour_slot']).reset_index(drop=True)
print(base.head())
print(f'Total slot-rows: {len(base)}')

        date  day_of_week  hour_slot  patient_count
0 2017-03-21            2          0              8
1 2017-03-21            2          1             11
2 2017-03-21            2          2             16
3 2017-03-21            2          3             21
4 2017-03-21            2          4             26
Total slot-rows: 6367


## 5. Build all 15 final features
Each one individually tested and validated earlier in the project.

In [ ]:
# --- Historical patient-flow features ---
# Each weekday + 4-hour slot occurs approximately once per week.
# Therefore, rolling N observations represent approximately N previous weeks.

for weeks in [2, 4, 8, 12]:
    base[f'lag_{weeks}wk_avg'] = (
        base.groupby(['day_of_week', 'hour_slot'])['patient_count']
        .transform(
            lambda x: x.shift(1).rolling(
                weeks,
                min_periods=1
            ).mean()
        )
    )

# --- Same-day momentum ---
base['daily_cumulative_avg'] = base.groupby('date')['patient_count'].transform(lambda x: x.expanding().mean().shift(1))
base['daily_cumulative_avg'] = base['daily_cumulative_avg'].fillna(base['lag_8wk_avg'])

# --- Calendar ---
base['is_weekend'] = base['day_of_week'].isin([0,6]).astype(int)
base['month'] = pd.to_datetime(base['date']).dt.month
base['week_of_year'] = pd.to_datetime(base['date']).dt.isocalendar().week.astype(int)

def add_slot_feature(df, source_col, agg='mean', newname=None):
    name = newname or f'avg_{source_col}'
    agg_df = (
        triage.groupby(['date','hour_slot'])[source_col]
        .agg(agg)
        .reset_index()
        .rename(columns={source_col: name})
    )
    df = df.merge(agg_df, on=['date','hour_slot'], how='left')
    lagname = f'lag_8wk_{name}'
    df[lagname] = df.groupby(['day_of_week','hour_slot'])[name].transform(lambda x: x.shift(1).rolling(8, min_periods=1).mean())
    return df, lagname

base, sev_col = add_slot_feature(base, 'TriageGrade', newname='severity')
base, age_col = add_slot_feature(base, 'age')
base, pain_col = add_slot_feature(base, 'PainGrade')
base, fast_col = add_slot_feature(base, 'NeedFastExecute')
base, crit_col = add_slot_feature(base, 'CriticalStatus')

# --- High-acuity rate (Grade 1-2) ---
triage['is_high_acuity'] = (triage['TriageGrade'] <= 2).astype(int)
base, acuity_col = add_slot_feature(base, 'is_high_acuity', newname='high_acuity')

# --- Complaint-type mix (4 macro categories from ICD chapter letters) ---
triage['icd_chapter'] = triage['ChiefComplaint'].astype(str).str[0]
category_map = {
    'S':'Trauma/Injury','T':'Trauma/Injury','W':'Trauma/Injury',
    'R':'Medical/Internal','K':'Medical/Internal','N':'Medical/Internal',
    'I':'Medical/Internal','G':'Medical/Internal','E':'Medical/Internal','J':'Medical/Internal',
    'M':'Musculoskeletal/Skin','L':'Musculoskeletal/Skin',
    'Z':'Other/Administrative'
}
triage['complaint_macro'] = triage['icd_chapter'].map(category_map).fillna('Other/Administrative')

complaint_mix = (
    triage.groupby(['date','hour_slot','complaint_macro'])
    .size().unstack(fill_value=0).reset_index()
)
base = base.merge(complaint_mix, on=['date','hour_slot'], how='left')
for cat in ['Medical/Internal','Trauma/Injury']:  # the 2 categories kept in the final 15-feature set
    base[f'lag_8wk_{cat}'] = base.groupby(['day_of_week','hour_slot'])[cat].transform(lambda x: x.shift(1).rolling(8, min_periods=1).mean())

base = base.sort_values(['date','hour_slot']).reset_index(drop=True)
print('Feature build complete.')
print(base.columns.tolist())

Feature build complete.
['date', 'day_of_week', 'hour_slot', 'patient_count', 'lag_8wk_avg', 'daily_cumulative_avg', 'is_weekend', 'month', 'week_of_year', 'severity', 'lag_8wk_severity', 'avg_age', 'lag_8wk_avg_age', 'avg_PainGrade', 'lag_8wk_avg_PainGrade', 'avg_NeedFastExecute', 'lag_8wk_avg_NeedFastExecute', 'avg_CriticalStatus', 'lag_8wk_avg_CriticalStatus', 'high_acuity', 'lag_8wk_high_acuity', 'Medical/Internal', 'Musculoskeletal/Skin', 'Other/Administrative', 'Trauma/Injury', 'lag_8wk_Medical/Internal', 'lag_8wk_Trauma/Injury']


## 6. Train final deployable model (on ALL data — this is what gets saved)
The model is first evaluated using chronological walk-forward validation to measure performance on unseen future observations.

After completing the walk-forward evaluation, the final deployable Random Forest and Neural Network models are retrained using all available historical observations. These models are then saved for deployment.

In [ ]:
feature_cols_15db = [
    'hour_slot', 'day_of_week', 'month', 'week_of_year', 'is_weekend',
    'lag_2wk_avg', 'lag_4wk_avg', 'lag_8wk_avg', 'lag_12wk_avg',
    'lag_8wk_severity', 'lag_8wk_high_acuity', 'daily_cumulative_avg',
    'lag_8wk_avg_PainGrade',
    'lag_8wk_Medical/Internal', 'lag_8wk_Trauma/Injury'
]
print(f'Total features: {len(feature_cols_15db)}')

# Evaluate with walk-forward, same as your official methodology
def walk_forward_15db(data, feature_cols, target_col, initial_train_frac=0.8, step_size=126):
    data = data.sort_values(['date','hour_slot']).reset_index(drop=True)
    data = data.dropna(subset=feature_cols).reset_index(drop=True)
    n = len(data)
    train_end = int(n * initial_train_frac)

    all_actuals, all_preds = [], []
    current_train_end = train_end
    while current_train_end < n:
        train_data = data.iloc[:current_train_end]
        step_end = min(current_train_end + step_size, n)
        test_step = data.iloc[current_train_end:step_end]

        X_tr, y_tr = train_data[feature_cols], train_data[target_col]
        X_te, y_te = test_step[feature_cols], test_step[target_col]

        scaler_wf = StandardScaler()
        X_tr_scaled = scaler_wf.fit_transform(X_tr)
        X_te_scaled = scaler_wf.transform(X_te)

        rf_wf = RandomForestRegressor(n_estimators=200, max_depth=6, criterion='absolute_error', random_state=42)
        rf_wf.fit(X_tr, y_tr)
        mlp_wf = MLPRegressor(hidden_layer_sizes=(50,), alpha=0.0001, learning_rate_init=0.01, max_iter=300, random_state=42, early_stopping=True, n_iter_no_change=5)
        mlp_wf.fit(X_tr_scaled, y_tr)

        preds = (rf_wf.predict(X_te) + mlp_wf.predict(X_te_scaled)) / 2
        all_actuals.extend(y_te.values)
        all_preds.extend(preds)
        current_train_end = step_end

    all_actuals, all_preds = np.array(all_actuals), np.array(all_preds)
    mae = mean_absolute_error(all_actuals, all_preds)
    rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
    nonzero_mask = all_actuals != 0

    mape = np.mean(
        np.abs(
            (all_actuals[nonzero_mask] - all_preds[nonzero_mask])
            / all_actuals[nonzero_mask]
        )
    ) * 100
    peak_thresh = np.quantile(all_actuals, 0.9)
    peak_mask = all_actuals >= peak_thresh
    peak_mae = mean_absolute_error(all_actuals[peak_mask], all_preds[peak_mask])
    print(f'Walk-forward (15 DB-compatible features) — MAE: {mae:.2f}, RMSE: {rmse:.2f}, MAPE: {mape:.2f}%, Peak MAE: {peak_mae:.2f}')

walk_forward_15db(base, feature_cols_15db, 'patient_count')

# Now train the actual deployable model on ALL data
X_all_db = base.dropna(subset=feature_cols_15db)[feature_cols_15db]
y_all_db = base.dropna(subset=feature_cols_15db)['patient_count']

scaler_db = StandardScaler()
X_all_db_scaled = scaler_db.fit_transform(X_all_db)

rf_db = RandomForestRegressor(n_estimators=200, max_depth=6, criterion='absolute_error', random_state=42)
rf_db.fit(X_all_db, y_all_db)

mlp_db = MLPRegressor(hidden_layer_sizes=(50,), alpha=0.0001, learning_rate_init=0.01, max_iter=2000, random_state=42, early_stopping=True)
mlp_db.fit(X_all_db_scaled, y_all_db)

print('Final deployable model trained on all data.')

Final deployable model trained on all available real data.


## 7. Save model and historical data for the backend

In [ ]:
joblib.dump({'rf': rf_db, 'mlp': mlp_db, 'scaler': scaler_db}, 'flow_prediction_model.pkl')

raw_cols = ['date','day_of_week','hour_slot','patient_count','severity','high_acuity',
            'avg_PainGrade','Medical/Internal','Trauma/Injury']
base[[c for c in raw_cols if c in base.columns]].to_csv('historical_flow_raw.csv', index=False)

files.download('flow_prediction_model.pkl')
files.download('historical_flow_raw.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>